In [ ]:
from scipy.stats import entropy
from scipy.special import rel_entr
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
import random
import numpy as np
import torch
import gdown

In [ ]:
import pickle
from sklearn.metrics import roc_auc_score

# Logit URLs and Paths.

This is the only cell to be changed when swapping between different datasets

## Imagenet

In [ ]:
base_url = 'https://drive.google.com/uc?id='


clean_train_min = base_url + '1mqGyx_tqnlrxX3JKftWmxq4KzJ5aCoLf'
poison_train_min = base_url + '1seswMPWJ4mlxxTqA24JIdPIeEpwC7emm'
clean_train_max = base_url + '1qDAw8pImTRNB-l8txrNC8X5MOXtah1Wr'
poison_train_max = base_url + '1jU8pHDCAXgNAHHgPnX2OHdiDOQp2odvN'


clean_test_min = base_url + '1QhzJv6Mbd0dJ0NMB7Od9HrTI67UgAPcI'
poison_test_min = base_url + '1SaLZ6dpg3a8F1SsJdz0HTr-n-F5eiq1P'
clean_test_max = base_url + '18tn5SokPljlGtF3moZpCIzU0jQvNkGXF'
poison_test_max = base_url + '1uMFPEv534GeeRCA5rFjldyHW8wXHNWMD'



gdown.download(clean_train_min)
gdown.download(poison_train_min)
gdown.download(clean_train_max)
gdown.download(poison_train_max)
gdown.download(clean_test_min)
gdown.download(poison_test_min)
gdown.download(clean_test_max)
gdown.download(poison_test_max)



train_clean_min_path = '/content/imagenet_clean_logits.pkl'
train_poison_min_path = '/content/imagenet_poisoned_logits.pkl'
train_clean_max_path = '/content/imagenet_clean_logits_max.pkl'
train_poison_max_path = '/content/imagenet_poisoned_logits_max.pkl'

test_clean_min_path = '/content/imagenet_clean_logits_test.pkl'
test_poison_min_path = '/content/imagenet_poisoned_logits_test.pkl'
test_clean_max_path = '/content/imagenet_clean_logits_max_test.pkl'
test_poison_max_path = '/content/imagenet_poisoned_logits_max_test.pkl'

Downloading...
From (original): https://drive.google.com/uc?id=1mqGyx_tqnlrxX3JKftWmxq4KzJ5aCoLf
From (redirected): https://drive.google.com/uc?id=1mqGyx_tqnlrxX3JKftWmxq4KzJ5aCoLf&confirm=t&uuid=29851b3e-bf2d-42d1-857f-1bfbda213be8
To: /content/imagenet_clean_logits.pkl
100%|██████████| 2.00G/2.00G [00:32<00:00, 61.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1seswMPWJ4mlxxTqA24JIdPIeEpwC7emm
From (redirected): https://drive.google.com/uc?id=1seswMPWJ4mlxxTqA24JIdPIeEpwC7emm&confirm=t&uuid=3ac303fe-b7bc-432a-a32c-61186ee6c804
To: /content/imagenet_poisoned_logits.pkl
100%|██████████| 2.00G/2.00G [00:27<00:00, 73.3MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1qDAw8pImTRNB-l8txrNC8X5MOXtah1Wr
From (redirected): https://drive.google.com/uc?id=1qDAw8pImTRNB-l8txrNC8X5MOXtah1Wr&confirm=t&uuid=b07b5324-a635-4436-b801-9455eb4505e7
To: /content/imagenet_clean_logits_max.pkl
100%|██████████| 2.00G/2.00G [01:31<00:00, 22.0MB/s]
Downloading...
Fro

## Tiny Imagenet

In [ ]:
base_url = 'https://drive.google.com/uc?id='

all_zipped = '1oeUigoG18APiZjLbJuhWeHEF1LeT_nHP'


gdown.download(base_url+all_zipped)

# unzip file
!unzip tiny.zip -d ./


train_clean_min_path = '/content/tiny_clean_min_train_logits.pkl'
train_poison_min_path = '/content/tiny_poisoned_min_train_logits.pkl'
train_clean_max_path = '/content/tiny_clean_max_train_logits.pkl'
train_poison_max_path = '/content/tiny_poisoned_max_train_logits.pkl'

test_clean_min_path = '/content/tiny_clean_min_test_logits.pkl'
test_poison_min_path = '/content/tiny_poisoned_min_test_logits.pkl'
test_clean_max_path = '/content/tiny_clean_max_test_logits.pkl'
test_poison_max_path = '/content/tiny_poisoned_max_test_logits.pkl'

Downloading...
From (original): https://drive.google.com/uc?id=1oeUigoG18APiZjLbJuhWeHEF1LeT_nHP
From (redirected): https://drive.google.com/uc?id=1oeUigoG18APiZjLbJuhWeHEF1LeT_nHP&confirm=t&uuid=6275f227-302a-41e6-b4fc-66f2d4b43744
To: /content/tiny.zip

  0%|          | 0.00/641M [00:00<?, ?B/s]
  0%|          | 2.62M/641M [00:00<01:36, 6.64MB/s]
  1%|          | 5.77M/641M [00:00<01:26, 7.36MB/s]
  1%|          | 7.86M/641M [00:00<01:11, 8.87MB/s]
  4%|▍         | 24.6M/641M [00:01<00:18, 33.0MB/s]
  4%|▍         | 28.8M/641M [00:01<00:27, 22.5MB/s]
  6%|▋         | 40.9M/641M [00:01<00:24, 24.9MB/s]
  8%|▊         | 49.8M/641M [00:02<00:18, 31.9MB/s]
 10%|▉         | 61.9M/641M [00:02<00:17, 33.0MB/s]
 10%|█         | 66.1M/641M [00:02<00:18, 31.4MB/s]
 11%|█         | 70.8M/641M [00:02<00:20, 28.2MB/s]
 13%|█▎        | 84.9M/641M [00:03<00:14, 38.9MB/s]
 14%|█▍        | 92.3M/641M [00:03<00:17, 31.0MB/s]
 15%|█▌        | 99.1M/641M [00:03<00:16, 33.5MB/s]
 18%|█▊        | 113M/641

Archive:  tiny.zip
  inflating: ./tiny_clean_max_test_logits.pkl  
  inflating: ./tiny_clean_max_train_logits.pkl  
  inflating: ./tiny_clean_min_test_logits.pkl  
  inflating: ./tiny_clean_min_train_logits.pkl  
  inflating: ./tiny_poisoned_max_test_logits.pkl  
  inflating: ./tiny_poisoned_max_train_logits.pkl  
  inflating: ./tiny_poisoned_min_test_logits.pkl  
  inflating: ./tiny_poisoned_min_train_logits.pkl  


## CIFAR-10

In [ ]:
base_url = 'https://drive.google.com/uc?id='


clean_train_min = base_url + '1_nRyX_mK69u95iuqcBWh-V4tEZ1syA-V'
poison_train_min = base_url + '1EACBnWhqB_4qhhJt1yXlXyZGlBiDaRzn'
clean_train_max = base_url + '1jySfnLTdTg3oidiHbd2OzzJmFOJCDQ8j'
poison_train_max = base_url + '1hx0KqcGQIYDJ8tYcgJjq35wCNAAa3VhJ'


clean_test_min = base_url + '11o7g69dpSvOLc96_Wg1G9003M9NtyqGs'
poison_test_min = base_url + '1U5vljmjPgCZ_b7cEjhgV97on-xo8AaMd'
clean_test_max = base_url + '1_sh546aXQrTtG0luS2Z1H_JMrIEqKJpV'
poison_test_max = base_url + '1cZCKl_Z5i9Iv67qPS7jP-GdwzuQ3nDTN'



gdown.download(clean_train_min)
gdown.download(poison_train_min)
gdown.download(clean_train_max)
gdown.download(poison_train_max)
gdown.download(clean_test_min)
gdown.download(poison_test_min)
gdown.download(clean_test_max)
gdown.download(poison_test_max)



train_clean_min_path = '/content/cifar_clean_min_train_logits.pkl'
train_poison_min_path = '/content/cifar_poisoned_min_train_logits.pkl'
train_clean_max_path = '/content/cifar_clean_max_train_logits.pkl'
train_poison_max_path = '/content/cifar_poisoned_max_train_logits.pkl'

test_clean_min_path = '/content/cifar_clean_min_test_logits.pkl'
test_poison_min_path = '/content/cifar_poisoned_min_test_logits.pkl'
test_clean_max_path = '/content/cifar_clean_max_test_logits.pkl'
test_poison_max_path = '/content/cifar_poisoned_max_test_logits.pkl'

Downloading...
From: https://drive.google.com/uc?id=1_nRyX_mK69u95iuqcBWh-V4tEZ1syA-V
To: /content/cifar_clean_min_train_logits.pkl

100%|██████████| 342k/342k [00:00<00:00, 43.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1EACBnWhqB_4qhhJt1yXlXyZGlBiDaRzn
To: /content/cifar_poisoned_min_train_logits.pkl

100%|██████████| 342k/342k [00:00<00:00, 72.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1jySfnLTdTg3oidiHbd2OzzJmFOJCDQ8j
To: /content/cifar_clean_max_train_logits.pkl

100%|██████████| 342k/342k [00:00<00:00, 65.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1hx0KqcGQIYDJ8tYcgJjq35wCNAAa3VhJ
To: /content/cifar_poisoned_max_train_logits.pkl

100%|██████████| 342k/342k [00:00<00:00, 72.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=11o7g69dpSvOLc96_Wg1G9003M9NtyqGs
To: /content/cifar_clean_min_test_logits.pkl

100%|██████████| 68.5k/68.5k [00:00<00:00, 30.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1U5vljmjPgCZ_b7cEjhgV

# Utils


In [ ]:
# Function to load and convert tensors to float16
def load_and_convert(filepath):
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    return {k: v.to(torch.float16) for k, v in data.items()}  # Convert tensors to float16



class Evaluator:
  @staticmethod
  def get_accuracy(y_true, y_pred):
    return accuracy_score(y_true, y_pred)

  @staticmethod
  def get_confusion_matrix(y_true, y_pred):
    return confusion_matrix(y_true, y_pred)

  @staticmethod
  def get_auc(y_true, y_anomaly_scores):
    return roc_auc_score(y_true, y_anomaly_scores)

  @staticmethod
  def get_f1_score(y_true, y_pred):
    return f1_score(y_true, y_pred)

# Detectors

In [ ]:
class GaussianDetector:
    """
    A detector that computes anomaly scores based on Gaussian log probability.

    Configuration parameters:
      - method: 'voting' or 'diagonal'
            (voting: uses softmax over the entire logit matrix,
             diagonal: uses only the main-diagonal entries)
      - standardized: bool, whether to apply standardization to the logits.

    The detector first estimates the mean and std vectors (the "clean signature") from a set of clean models.
    It can then compute an anomaly score for each new model and determine an optimal threshold by comparing
    known clean and poisoned models.
    """
    def __init__(self, method='voting', standardized=False):
        assert method in ['voting', 'diagonal'], f"Invalid method: {method}"
        self.method = method
        self.standardized = standardized

        # Will be set via set_parameters()
        self.clean_signature_means = None
        self.clean_signature_stds = None
        self.optimal_threshold = None

    def compute_feature(self, logits):
        """
        Computes the feature vector for a given logit matrix.

        For method 'voting':
          - If standardized: each row is standardized before applying softmax,
            then the probabilities are averaged over rows.
          - Else: directly apply softmax and take the mean over rows.

        For method 'diagonal':
          - If standardized: standardize the diagonal entries.
          - Else: simply extract the diagonal as the feature.
        """
        eps = 1e-7
        if self.method == 'voting':
            if self.standardized:
                # Standardize each row before softmax.
                logits_std = (logits - logits.mean(dim=1, keepdim=True)) / (logits.std(dim=1, keepdim=True) + eps)
                feature = torch.softmax(logits_std, dim=1).mean(dim=0)
            else:
                feature = torch.softmax(logits, dim=1).mean(dim=0)
        elif self.method == 'diagonal':
            # Extract the diagonal entries (convert to float32 for numerical stability)
            raw_diag = torch.diagonal(logits, dim1=0, dim2=1).to(torch.float32)
            if self.standardized:
                feature = (raw_diag - raw_diag.mean()) / (raw_diag.std() + eps)
            else:
                feature = raw_diag
        else:
            raise ValueError("Invalid method selected.")
        return feature

    def set_parameters(self, clean_logits_tensor):
        """
        Estimates the clean signature parameters from a tensor of clean logits.

        clean_logits_tensor: tensor of shape (num_models, num_classes, num_classes)
        """
        features = []
        for i in range(clean_logits_tensor.size(0)):
            feat = self.compute_feature(clean_logits_tensor[i])
            features.append(feat)
        features = torch.stack(features)  # (num_models, num_classes)
        self.clean_signature_means = features.mean(dim=0)
        self.clean_signature_stds = features.std(dim=0)

    def compute_log_prob(self, feature_vector):
        """
        Computes the log probability of a feature vector under the global Gaussian signature.
        """
        eps = 1e-7
        stds = self.clean_signature_stds.to(torch.float32) + eps
        means = self.clean_signature_means.to(torch.float32)
        # Gaussian log probability (up to a constant)
        log_probs = -0.5 * torch.log(2 * torch.pi * (stds**2)) - ((feature_vector - means)**2 / (2 * stds**2))
        return log_probs.sum()

    def compute_anomaly(self, logits):
        """
        Computes the anomaly score (negative log probability) for given logits.
        """
        feature = self.compute_feature(logits)
        return -self.compute_log_prob(feature)



    def get_optimal_threshold(self, model_instances, percentile=95.0):
        """
        Compute a threshold based on a given percentile of the anomaly scores for normal objects.
        Parameters:
            model_instances: list of InputModel objects.
            percentile: float, the percentile value to use on the normal scores (default 95.0).
        Returns:
            (accuracy, threshold): The accuracy computed on the entire set using this threshold and the threshold value itself.
        Note: In this context, an instance is considered 'poison' (anomaly) if its anomaly score exceeds the threshold.
              The threshold is chosen based solely on the normal (non-poison) objects.
        """
        # Extract anomaly scores and labels (1 if 'poison', 0 if 'normal')
        scores = torch.tensor([model.anomaly_score for model in model_instances])
        labels = torch.tensor([1 if model.model_type == 'poison' else 0 for model in model_instances])

        # Select scores of normal instances (label == 0)
        normal_scores = scores[labels == 0]

        # Calculate the threshold as the given percentile of the normal scores distribution.
        threshold = np.percentile(normal_scores.numpy(), percentile)

        # Classify as 'poison' if the score exceeds the threshold.
        preds = (scores > threshold).int()

        # Compute accuracy over the entire set.
        accuracy = (preds == labels).float().mean().item()

        # Save and return the computed threshold and accuracy.
        self.optimal_threshold = threshold
        return accuracy, threshold



class InputModel:
    """
    A lightweight model wrapper that computes its anomaly score using a given detector.

    Parameters:
       - model_id: an identifier for the model
       - logits: tensor of shape (num_classes, num_classes)
       - model_type: string, either 'clean' or 'poison'
       - detector: an instance of GaussianDetector, used to compute the anomaly score.
    """
    def __init__(self, model_id, logits, model_type, detector: GaussianDetector):
        assert model_type in ['clean', 'poison'], f'Invalid model type: {model_type}'
        self.model_id = model_id
        self.model_type = model_type
        self.anomaly_score = detector.compute_anomaly(logits)



def train_detector(detector, train_clean_path, num_clean_models, percentile, random_seed=42):
    """
    Trains the detector using a subset of clean logits from training data.
    The detector's threshold is estimated from a given percentile of anomaly scores computed
    from the randomly sampled clean models.

    Parameters:
        detector: an instance of GaussianDetector.
        train_clean_path: filepath for clean training data.
        num_clean_models: integer, number of clean models to randomly sample from the loaded data.
        percentile: float, percentile to use for setting the anomaly threshold (e.g., 95.0).
        random_seed: 42 by default.

    Returns:
        anomaly_scores_dict: dictionary containing the anomaly scores for the clean models.
    """
    # Load clean logits.
    clean_logits = load_and_convert(train_clean_path)

    # Set seed for reproducibility.
    np.random.seed(random_seed)

    # Randomly sample keys from the clean logits.
    keys = list(clean_logits.keys())
    sampled_keys = np.random.choice(keys, size=num_clean_models, replace=False)

    # Use the sampled clean logits to set global detector parameters.
    selected_clean_tensors = [clean_logits[key] for key in sampled_keys]
    clean_tensor = torch.stack(selected_clean_tensors)
    detector.set_parameters(clean_tensor)

    # Construct training models using only the sampled clean models.
    training_models = []
    for key in sampled_keys:
        training_models.append(InputModel(key, clean_logits[key], 'clean', detector))

    # Optimize threshold using the specified percentile.
    train_acc, optimal_threshold = detector.get_optimal_threshold(training_models, percentile)

    # Evaluate performance on the sampled clean models.
    y_anomaly_scores = []
    anomaly_scores_dict = {'clean': []}

    for m in training_models:
        y_anomaly_scores.append(m.anomaly_score)
        anomaly_scores_dict['clean'].append(m.anomaly_score)


    anomaly_scores_dict['clean'] = np.array(anomaly_scores_dict['clean'])
    return anomaly_scores_dict


def test_detector(detector, test_clean_path, test_poison_path):
    """
    Evaluates the detector on test data.

    detector: an instance of GaussianDetector with parameters and threshold previously set.
    test_clean_path, test_poison_path: filepaths for test data.

    Returns a dictionary of anomaly scores separated by model type.
    """
    # Load test data.
    clean_logits = load_and_convert(test_clean_path)
    poison_logits = load_and_convert(test_poison_path)

    test_models = []
    for model_id in clean_logits:
        test_models.append(InputModel(model_id, clean_logits[model_id], 'clean', detector))
    for model_id in poison_logits:
        test_models.append(InputModel(model_id, poison_logits[model_id], 'poison', detector))

    y_true = []
    y_pred = []
    y_anomaly_scores = []
    anomaly_scores_dict = {'clean': [], 'poison': []}

    for m in test_models:
        y_true.append(1 if m.model_type == 'poison' else 0)
        y_pred.append(1 if m.anomaly_score > detector.optimal_threshold else 0)
        y_anomaly_scores.append(m.anomaly_score)
        anomaly_scores_dict[m.model_type].append(m.anomaly_score)

    acc = Evaluator.get_accuracy(y_true, y_pred)
    conf_matrix = Evaluator.get_confusion_matrix(y_true, y_pred)
    auc = Evaluator.get_auc(y_true, y_anomaly_scores)
    f1_score = Evaluator.get_f1_score(y_true, y_pred)
    # print(f'Test Accuracy: {acc}')
    # print(f'Confusion Matrix:\n{conf_matrix}')
    # print(f'AUC: {auc}')
    # print(f'F1 Score: {f1_score}')

    anomaly_scores_dict['clean'] = np.array(anomaly_scores_dict['clean'])
    anomaly_scores_dict['poison'] = np.array(anomaly_scores_dict['poison'])
    return anomaly_scores_dict, acc, auc



# MinMax Ensemble

In [ ]:
class MinMaxEnsembler:
  optimal_threshold = None

  def __init__(self, method='sum scores', standardized=False):
        assert method in ['sum scores'], f"Invalid method: {method}"
        self.method = method


  def get_optimal_threshold(self, min_anomaly_scores_dict, max_anomaly_scores_dict, percentile=95.0):
      """
      Compute a threshold based on a given percentile of the anomaly scores for normal objects.
      Parameters:
          min_anomaly_scores_dict: {'clean': (num_models,), 'poison': (num_models,)}
          max_anomaly_scores_dict: {'clean': (num_models,), 'poison': (num_models,)}
          percentile: float, the percentile value to use on the normal scores (default 95.0).
      Returns:
          (accuracy, threshold): The accuracy computed on the entire set using this threshold and the threshold value itself.
      Note: In this context, an instance is considered 'poison' (anomaly) if its anomaly score exceeds the threshold.
            The threshold is chosen based solely on the normal (non-poison) objects.
      """
      # Extract anomaly scores and labels (1 if 'poison', 0 if 'normal')
      if self.method == 'sum scores':
        clean_summed_scores = min_anomaly_scores_dict['clean'] + max_anomaly_scores_dict['clean']

      # Calculate the threshold as the given percentile of the normal scores distribution.
      threshold = np.percentile(clean_summed_scores, percentile)

      # Save and return the computed threshold and accuracy.
      self.optimal_threshold = threshold
      return threshold


def train_both(ensembler, train_min_anomaly_scores, train_max_anomaly_scores):
  ensembler.get_optimal_threshold(train_min_anomaly_scores, train_max_anomaly_scores)
  num_models = len(train_min_anomaly_scores['clean'])
  y_anomaly_scores = []

  for model_type in train_min_anomaly_scores.keys():
    for model_idx in range(num_models):
      y_anomaly_scores.append(train_min_anomaly_scores[model_type][model_idx] + train_max_anomaly_scores[model_type][model_idx])

  return y_anomaly_scores




def test_both(ensembler, test_min_anomaly_scores, test_max_anomaly_scores):
  num_models = len(test_min_anomaly_scores['clean'])
  y_true = []
  y_pred = []
  y_anomaly_scores = []

  for model_type in ['clean', 'poison']:
    for model_idx in range(num_models):
      y_true.append(1 if model_type == 'poison' else 0)
      y_anomaly_scores.append(test_min_anomaly_scores[model_type][model_idx] + test_max_anomaly_scores[model_type][model_idx])

  y_true = np.array(y_true).astype(int)
  y_anomaly_scores = np.array(y_anomaly_scores)
  y_pred = (y_anomaly_scores > ensembler.optimal_threshold).astype(int)

  acc = Evaluator.get_accuracy(y_true, y_pred)
  conf_matrix = Evaluator.get_confusion_matrix(y_true, y_pred)
  auc = Evaluator.get_auc(y_true, y_anomaly_scores)
  f1_score = Evaluator.get_f1_score(y_true, y_pred)
  # print(f'Test Accuracy: {acc}')
  # print(f'Confusion Matrix:\n{conf_matrix}')
  # print(f'AUC: {auc}')

  return acc, auc


# Experiment

## Experiment Array

In [ ]:
import random
from tqdm.auto import tqdm

random.seed(42)
seeds = random.sample(range(1, 10000), 5)
print("Seeds: ", seeds)

num_clean_models_arr = [50, 100, 200, 300]
print("num_clean_models: ", num_clean_models_arr)

percentiles = [95]
print("percentiles: ", percentiles)

num_detectors = 3 # Min, Max, and Both

accuracy_array = np.zeros((len(seeds), len(percentiles),len(num_clean_models_arr), num_detectors))
auc_array = np.zeros((len(seeds), len(percentiles),len(num_clean_models_arr), num_detectors))

print(f"Acc/AUC Array Shapes: {accuracy_array.shape}")



for seed_idx, seed in tqdm(enumerate(seeds), desc="Seed:" , total=len(seeds)):
  for percentile_idx, percentile in tqdm(enumerate(percentiles), desc="Percentile:" , total=len(percentiles)):
    for num_clean_models_idx, num_clean_models in tqdm(enumerate(num_clean_models_arr), desc="Num Clean:" , total=len(num_clean_models_arr)):
      min_detector = GaussianDetector(method='voting', standardized=True)
      max_detector = GaussianDetector(method='diagonal', standardized=True)
      min_max_ensemble = MinMaxEnsembler()
      min_train_scores = train_detector(min_detector, train_clean_min_path, num_clean_models = num_clean_models, percentile=percentile, random_seed=seed)
      max_train_scores = train_detector(max_detector, train_clean_max_path, num_clean_models = num_clean_models, percentile=percentile, random_seed=seed)
      train_both(min_max_ensemble, min_train_scores, max_train_scores)

      min_test_scores, min_acc, min_auc = test_detector(min_detector, test_clean_min_path, test_poison_min_path)
      max_test_scores, max_acc, max_auc = test_detector(max_detector, test_clean_max_path, test_poison_max_path)
      min_max_acc, min_max_auc = test_both(min_max_ensemble, min_test_scores, max_test_scores)

      accuracy_array[seed_idx, percentile_idx, num_clean_models_idx, 0] = min_acc
      accuracy_array[seed_idx, percentile_idx, num_clean_models_idx, 1] = max_acc
      accuracy_array[seed_idx, percentile_idx, num_clean_models_idx, 2] = min_max_acc

      auc_array[seed_idx, percentile_idx, num_clean_models_idx, 0] = min_auc
      auc_array[seed_idx, percentile_idx, num_clean_models_idx, 1] = max_auc
      auc_array[seed_idx, percentile_idx, num_clean_models_idx, 2] = min_max_auc





Seeds:  [1825, 410, 4507, 4013, 3658]
num_clean_models:  [50, 100, 200, 300]
percentiles:  [95]
Acc/AUC Array Shapes: (5, 1, 4, 3)


Seed::   0%|          | 0/5 [00:00<?, ?it/s]

Percentile::   0%|          | 0/1 [00:00<?, ?it/s]

Num Clean::   0%|          | 0/4 [00:00<?, ?it/s]

Percentile::   0%|          | 0/1 [00:00<?, ?it/s]

...

Num Clean::   0%|          | 0/4 [00:00<?, ?it/s]

Percentile::   0%|          | 0/1 [00:00<?, ?it/s]

Num Clean::   0%|          | 0/4 [00:00<?, ?it/s]

Percentile::   0%|          | 0/1 [00:00<?, ?it/s]

Num Clean::   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
print(accuracy_array)
print(auc_array)

[[[[0.73  0.655 0.665]
   [0.765 0.745 0.8  ]
   [0.77  0.755 0.825]
   [0.765 0.75  0.81 ]]]


 [[[0.745 0.77  0.74 ]
   [0.8   0.75  0.82 ]
   [0.78  0.76  0.825]
   [0.775 0.735 0.83 ]]]


 [[[0.785 0.665 0.625]
   [0.795 0.745 0.82 ]
   [0.785 0.73  0.83 ]
   [0.78  0.715 0.82 ]]]


 [[[0.72  0.7   0.64 ]
   [0.755 0.76  0.795]
   [0.765 0.755 0.825]
   [0.775 0.74  0.825]]]


 [[[0.77  0.725 0.755]
   [0.79  0.745 0.825]
   [0.775 0.72  0.82 ]
   [0.76  0.73  0.815]]]]
[[[[0.8578 0.8327 0.8996]
   [0.8602 0.8374 0.8998]
   [0.8679 0.8372 0.9109]
   [0.8671 0.8406 0.9123]]]


 [[[0.8661 0.8322 0.9152]
   [0.8697 0.8441 0.9151]
   [0.8709 0.8417 0.9165]
   [0.8701 0.8471 0.9153]]]


 [[[0.8598 0.8236 0.9036]
   [0.8652 0.8299 0.912 ]
   [0.872  0.8377 0.9175]
   [0.8727 0.8456 0.917 ]]]


 [[[0.8328 0.827  0.8858]
   [0.8419 0.8412 0.8975]
   [0.8629 0.8441 0.9112]
   [0.8673 0.8403 0.9113]]]


 [[[0.8668 0.8282 0.9129]
   [0.8695 0.8392 0.9133]
   [0.8697 0.8411 0.9134]
   [0.8688 

## Save Arr

In [ ]:
import pickle

# Save accuracy_array and auc_array
with open('accuracy_array.pkl', 'wb') as f:
    pickle.dump(accuracy_array, f)

with open('auc_array.pkl', 'wb') as f:
    pickle.dump(auc_array, f)


## Load Arr

In [ ]:
import pickle

# Load Array
with open('accuracy_array.pkl', 'rb') as f:
    accuracy_array = pickle.load(f)

with open('auc_array.pkl', 'rb') as f:
    auc_array = pickle.load(f)

## Calculate (Meand and Std.)

In [ ]:
# Arrays of shape: (Num Seeds, Num Percentiles, Num Clean Models, 3 [min, max, both])


# Draw a Plot for every percentile

means_acc = np.mean(accuracy_array, axis=0) # (Num Percentiles, Num Clean Models, 3)
stds_acc = np.std(accuracy_array, axis=0)

means_auc = np.mean(auc_array, axis=0)
stds_auc = np.std(auc_array, axis=0)




In [ ]:
print(means_acc)
print(stds_acc)

print(means_auc)
print(stds_auc)

[[[0.75  0.703 0.685]
  [0.781 0.749 0.812]
  [0.775 0.744 0.825]
  [0.771 0.734 0.82 ]]]
[[[0.02428992 0.04178516 0.05282045]
  [0.01772005 0.00583095 0.01208305]
  [0.00707107 0.01593738 0.00316228]
  [0.00734847 0.01157584 0.00707107]]]
[[[0.85666 0.82874 0.90342]
  [0.8613  0.83836 0.90754]
  [0.86868 0.84036 0.9139 ]
  [0.8692  0.84264 0.91374]]]
[[[0.01242893 0.00338798 0.01052357]
  [0.01030126 0.00477937 0.00736114]
  [0.00319399 0.00258426 0.00269295]
  [0.00206107 0.00308325 0.00209628]]]


## Plot Results (Save Figure)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_error_bars(ax, means, stds, x_tick_labels, model_type, ylabel):
    x = np.arange(len(x_tick_labels))
    lower = stds
    upper = stds
    yerr = [lower, upper]

    ax.errorbar(
        x, means,
        yerr=yerr, fmt='o-', capsize=4, color='blue',
        markerfacecolor='blue', lw=1.2, mec='blue'
    )
    ax.set_xticks(x)
    ax.set_xticklabels(x_tick_labels, rotation=45, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} ({model_type}) Model")
    ax.grid(True, linestyle='--', alpha=0.5)

def plot_percentile(
    p_idx,
    means_acc, stds_acc,
    means_auc, stds_auc,
    num_clean_labels,
    percentile_label=None,
    save_dir="."
):
    error_types = ['min', 'max', 'both']

    fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex='col')
    for col, et in enumerate(error_types):
        # Accuracy (top row)
        plot_error_bars(
            axes[0, col],
            means_acc[p_idx, :, col],
            stds_acc[p_idx, :, col],
            num_clean_labels,
            et,
            ylabel="Accuracy"
        )
        axes[0, col].tick_params(axis='x', labelbottom=True)
        axes[0, col].set_xlabel("Number of Clean Models")

        # AUC (bottom row)
        plot_error_bars(
            axes[1, col],
            means_auc[p_idx, :, col],
            stds_auc[p_idx, :, col],
            num_clean_labels,
            et,
            ylabel="AUC"
        )
        axes[1, col].set_xlabel("Number of Clean Models")

    # bump up y‐tick density on the Accuracy-"both" subplot
    ax_special = axes[0, 2]
    ax_special.yaxis.set_major_locator(MaxNLocator(nbins=12))
    ax_special.tick_params(axis='y', labelsize=12)

    title = (f"Clean Model {percentile_label}th Percentile Threshold"
             if percentile_label else f"Percentile index: {p_idx}")
    fig.suptitle(title, fontsize=16)

    plt.tight_layout(rect=[0, 0, 1, 0.95])

    # ——— Save to disk ———
    # build a safe filename
    label = percentile_label if percentile_label is not None else str(p_idx)
    fname = f"{label}_percentile.png"
    out_path = os.path.join(save_dir, fname)
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)   # free memory

    print(f"Saved percentile plot to {out_path}")


def plot_all_percentiles(
    means_acc, stds_acc,
    means_auc,  stds_auc,
    model_labels,
    percentile_labels=None,
    save_dir="."
):
    P = means_acc.shape[0]
    os.makedirs(save_dir, exist_ok=True)
    for p in range(P):
        label = percentile_labels[p] if percentile_labels is not None else None
        plot_percentile(
            p,
            means_acc, stds_acc,
            means_auc, stds_auc,
            model_labels,
            percentile_label=label,
            save_dir=save_dir
        )

# Example usage:

num_model_labels  = [str(k) for k in num_clean_models_arr]
percentile_labels = ["95"]

plot_all_percentiles(
    means_acc, stds_acc,
    means_auc,  stds_auc,
    num_model_labels,
    percentile_labels,
    save_dir="percentile_plots"
)


Saved percentile plot to percentile_plots/95_percentile.png


# Plot and Save (All three together)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_percentile(
    p_idx,
    means_acc, stds_acc,
    means_auc,  stds_auc,
    x_labels,
    percentile_label=None,
    save_dir=".",
    show_error_bars=True
):
    error_types = ['min', 'max', 'both']
    colors = ['tab:blue', 'tab:orange', 'tab:green']

    x = np.arange(len(x_labels))
    label = percentile_label if percentile_label is not None else str(p_idx)
    title = (f"Clean Model {percentile_label}th Percentile Threshold"
             if percentile_label else f"Percentile index: {p_idx}")

    fig, (ax_acc, ax_auc) = plt.subplots(1, 2, figsize=(12, 5), sharex=True)

    # Accuracy subplot
    for col, et in enumerate(error_types):
        y = means_acc[p_idx, :, col]
        if show_error_bars:
            yerr = [stds_acc[p_idx, :, col]] * 2
            ax_acc.errorbar(
                x, y, yerr=yerr,
                fmt='o-', capsize=4,
                color=colors[col], lw=1.2, mec=colors[col],
                markerfacecolor=colors[col], label=et
            )
        else:
            ax_acc.plot(
                x, y, 'o-', lw=1.2,
                color=colors[col],
                markerfacecolor=colors[col],
                mec=colors[col],
                label=et
            )
    ax_acc.set_title("ImageNet Results", fontsize=16)
    ax_acc.set_ylabel("Accuracy", fontsize=14)
    ax_acc.set_xticks(x)
    ax_acc.set_xticklabels(x_labels, rotation=45, ha='right')
    ax_acc.set_xlabel("Number of Clean Models", fontsize=14)
    ax_acc.grid(True, linestyle='--', alpha=0.5)
    ax_acc.legend(title="", fontsize=14, title_fontsize=16)
    ax_acc.yaxis.set_major_locator(MaxNLocator(nbins=12))

    # AUC subplot
    for col, et in enumerate(error_types):
        y = means_auc[p_idx, :, col]
        if show_error_bars:
            yerr = [stds_auc[p_idx, :, col]] * 2
            ax_auc.errorbar(
                x, y, yerr=yerr,
                fmt='o-', capsize=4,
                color=colors[col], lw=1.2, mec=colors[col],
                markerfacecolor=colors[col], label=et
            )
        else:
            ax_auc.plot(
                x, y, 'o-', lw=1.2,
                color=colors[col],
                markerfacecolor=colors[col],
                mec=colors[col],
                label=et
            )
    ax_auc.set_title("AUC")
    ax_auc.set_ylabel("AUC")
    ax_auc.set_xticks(x)
    ax_auc.set_xticklabels(x_labels, rotation=45, ha='right')
    ax_auc.set_xlabel("Number of Clean Models")
    ax_auc.grid(True, linestyle='--', alpha=0.5)
    ax_auc.legend(title="", fontsize=14, title_fontsize=16)

    fig.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.93])

    os.makedirs(save_dir, exist_ok=True)
    fname = f"{label}_percentile.png"
    out_path = os.path.join(save_dir, fname)
    fig.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

    print(f"Saved percentile plot to {out_path}")

def plot_all_percentiles(
    means_acc, stds_acc,
    means_auc,  stds_auc,
    model_labels,
    percentile_labels=None,
    save_dir=".",
    show_error_bars=False
):
    P = means_acc.shape[0]
    for p in range(P):
        label = percentile_labels[p] if percentile_labels is not None else None
        plot_percentile(
            p,
            means_acc, stds_acc,
            means_auc,  stds_auc,
            model_labels,
            percentile_label=label,
            save_dir=save_dir,
            show_error_bars=show_error_bars
        )



num_model_labels  = [str(k) for k in num_clean_models_arr]
percentile_labels = ["95"]

plot_all_percentiles(
    means_acc, stds_acc,
    means_auc,  stds_auc,
    num_model_labels,
    percentile_labels,
    save_dir="percentile_plots"
)

Saved percentile plot to percentile_plots/95_percentile.png


# Display Table

In [ ]:
!pip install tabulate
!pip install ace-tools


In [ ]:
import numpy as np
import pandas as pd

# Example arrays
a = means_acc[0]
b = stds_acc[0]

# build as a list of lists of formatted strings
rows, cols = a.shape
table = [
    [f"{a[i, j]:.3f} ± {b[i, j]:.3f}" for j in range(cols)]
    for i in range(rows)
]

# if you just want the raw Python structure:
print(table)

# or, for a neat tabular display, convert to a DataFrame:
df = pd.DataFrame(table,
                  index=[N for N in num_clean_models_arr],
                  columns=[model_type for model_type in ('Min', 'Max', 'Both')])
print(df)


[['0.876 ± 0.034', '0.912 ± 0.028', '0.869 ± 0.020'], ['0.945 ± 0.009', '0.942 ± 0.017', '0.917 ± 0.021'], ['0.962 ± 0.002', '0.949 ± 0.017', '0.944 ± 0.013'], ['0.964 ± 0.002', '0.954 ± 0.014', '0.952 ± 0.008']]
               Min            Max           Both
50   0.876 ± 0.034  0.912 ± 0.028  0.869 ± 0.020
100  0.945 ± 0.009  0.942 ± 0.017  0.917 ± 0.021
200  0.962 ± 0.002  0.949 ± 0.017  0.944 ± 0.013
300  0.964 ± 0.002  0.954 ± 0.014  0.952 ± 0.008
